# 🎓 Student Pass/Fail Machine Learning Project\n### Complete Solution for Exercises 1, 2, and 3 (All Tasks in One Folder)\n**Course Assignment:** Model Persistence with `joblib`, Flask API Deployment, and Streamlit Interface.

## 📦 Step 1: Install Libraries & Create Unified Project Folder

In [ ]:
!pip install scikit-learn joblib flask streamlit requests matplotlib pandas\n!mkdir -p student_prediction_project\n%cd student_prediction_project\nprint('✅ Project folder created!')

## 🏋️ Step 2: Task 1 - Train Model & Save with joblib (train_model.py)

In [ ]:
%%writefile train_model.py\n"""
train_model.py - Exercise 1: Model Persistence with joblib
Tasks:
1. Train a classification model (Logistic Regression) on student study hours & attendance.
2. Save the trained model to 'student_pass_model.joblib' using joblib.dump().
3. Load the saved model using joblib.load().
4. Make 3 predictions with different study hours.
5. Explain why retraining is not needed.
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import joblib

def train_and_save_model():
    print("=" * 60)
    print("STEP 1: Preparing Synthetic Student Training Dataset")
    print("=" * 60)
    
    # Dataset with study hours and attendance percentage
    data = {
        'study_hours': [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0, 8.5, 9.0],
        'attendance_percentage': [50, 55, 60, 65, 70, 60, 75, 80, 70, 85, 90, 85, 90, 95, 98],
        'passed': [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
    }
    
    df = pd.DataFrame(data)
    X = df[['study_hours', 'attendance_percentage']]
    y = df['passed']
    
    print(f"Dataset preview:\n{df.head()}")

    print("\n" + "=" * 60)
    print("STEP 2: Training Logistic Regression Model")
    print("=" * 60)
    model = LogisticRegression(random_state=42)
    model.fit(X, y)
    print("Model training complete.")
    print(f"Feature Coefficients (Weights): {model.coef_[0]}")
    print(f"Model Intercept (Bias): {model.intercept_[0]}")

    print("\n" + "=" * 60)
    print("STEP 3: Saving Model with joblib.dump()")
    print("=" * 60)
    model_filename = 'student_pass_model.joblib'
    joblib.dump(model, model_filename)
    print(f"Model successfully saved to '{model_filename}'!")

    print("\n" + "=" * 60)
    print("STEP 4: Loading Model with joblib.load()")
    print("=" * 60)
    loaded_model = joblib.load(model_filename)
    print("Model successfully loaded from disk.")

    print("\n" + "=" * 60)
    print("STEP 5: Making Three Predictions Using Different Study Hours")
    print("=" * 60)
    test_cases = [
        {"name": "Student A (Low Hours)", "study_hours": 1.5, "attendance": 55.0},
        {"name": "Student B (Average Hours)", "study_hours": 4.5, "attendance": 75.0},
        {"name": "Student C (High Hours)", "study_hours": 8.0, "attendance": 90.0}
    ]

    for idx, test in enumerate(test_cases, start=1):
        input_data = np.array([[test["study_hours"], test["attendance"]]])
        pred = loaded_model.predict(input_data)[0]
        prob = loaded_model.predict_proba(input_data)[0][1]
        status = "PASS" if pred == 1 else "FAIL"
        print(f"Prediction #{idx}: {test['name']}")
        print(f"  Inputs: Study Hours = {test['study_hours']} hrs | Attendance = {test['attendance']}%")
        print(f"  Result: {status} (Pass Probability: {prob:.2%})\n")

    print("=" * 60)
    print("EXPLANATION: Why the model does NOT need to be retrained:")
    print("=" * 60)
    explanation = (
        "During model training, scikit-learn computes optimal mathematical parameters\n"
        "(weights/coefficients and bias/intercept) from the dataset.\n"
        "Joblib serializes and stores these learned weights into 'student_pass_model.joblib'.\n"
        "When joblib.load() is called, those exact weights are loaded directly into memory.\n"
        "Inference simply evaluates the learned mathematical formula:\n"
        "    P(Pass) = 1 / (1 + e^-(w1 * hours + w2 * attendance + b))\n"
        "Because the parameters are already stored, predictions run instantly without\n"
        "retraining or needing the original dataset."
    )
    print(explanation)

if __name__ == '__main__':
    train_and_save_model()

In [ ]:
!python train_model.py

## 🌐 Step 3: Task 2 - Flask API Endpoint (app.py)

In [ ]:
%%writefile app.py\n"""
app.py - Exercise 2: Flask API for Student Pass/Fail Prediction
Tasks:
1. Accept both 'study_hours' and 'attendance_percentage'.
2. Validate attendance is between 0 and 100.
3. Return both values and model prediction in JSON.
4. Test with curl / Postman.
"""
from flask import Flask, request, jsonify
import joblib
import numpy as np
import os

app = Flask(__name__)

MODEL_PATH = os.path.join(os.path.dirname(__file__), 'student_pass_model.joblib')

try:
    model = joblib.load(MODEL_PATH)
    print(f"[*] Loaded model successfully from {MODEL_PATH}")
except Exception as e:
    print(f"[!] Warning: Model file not found. Run 'python train_model.py' first. Error: {e}")
    model = None

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        "message": "Student Pass/Fail Prediction Flask API is running.",
        "endpoints": {
            "/predict": "POST - JSON body: {'study_hours': float, 'attendance_percentage': float}",
            "/health": "GET - API health check"
        }
    })

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        "status": "healthy",
        "model_loaded": model is not None
    }), 200

@app.route('/predict', methods=['POST'])
def predict():
    global model
    if model is None:
        return jsonify({"error": "Model is not loaded. Train model first."}), 500

    data = request.get_json()
    if not data:
        return jsonify({"error": "Invalid request. Provide JSON payload."}), 400

    # 1. Check for required input fields
    if 'study_hours' not in data:
        return jsonify({"error": "Missing required field: 'study_hours'"}), 400
    if 'attendance_percentage' not in data:
        return jsonify({"error": "Missing required field: 'attendance_percentage'"}), 400

    # 2. Extract and parse numbers
    try:
        study_hours = float(data['study_hours'])
        attendance_percentage = float(data['attendance_percentage'])
    except (ValueError, TypeError):
        return jsonify({"error": "Inputs must be numeric numbers."}), 400

    # 3. Validate input bounds (0 to 100 for attendance, 0 to 24 for study hours)
    if study_hours < 0 or study_hours > 24:
        return jsonify({"error": "Validation Error: 'study_hours' must be between 0 and 24."}), 400

    if attendance_percentage < 0 or attendance_percentage > 100:
        return jsonify({"error": "Validation Error: 'attendance_percentage' must be between 0 and 100."}), 400

    # 4. Perform model prediction
    try:
        input_features = np.array([[study_hours, attendance_percentage]])
        raw_pred = int(model.predict(input_features)[0])
        probabilities = model.predict_proba(input_features)[0]
        pass_prob = float(probabilities[1])
        result_label = "Pass" if raw_pred == 1 else "Fail"

        # 5. Return JSON response
        return jsonify({
            "status": "success",
            "study_hours": study_hours,
            "attendance_percentage": attendance_percentage,
            "prediction": result_label,
            "pass_probability": round(pass_prob, 4),
            "confidence_percentage": round(pass_prob * 100, 2) if raw_pred == 1 else round((1 - pass_prob) * 100, 2)
        }), 200

    except Exception as e:
        return jsonify({"error": f"Internal prediction error: {str(e)}"}), 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)

In [ ]:
import subprocess, time\nflask_proc = subprocess.Popen(['python', 'app.py'])\ntime.sleep(3)\nprint('Flask API started on port 5000!')\n!curl -X POST http://127.0.0.1:5000/predict -H 'Content-Type: application/json' -d '{"study_hours": 7.0, "attendance_percentage": 85.0}'\n!curl -X POST http://127.0.0.1:5000/predict -H 'Content-Type: application/json' -d '{"study_hours": 5.0, "attendance_percentage": 120.0}'

## 💻 Step 4: Task 2 (Part B) - Streamlit Interface (streamlit_app.py)

In [ ]:
%%writefile streamlit_app.py\n"""
streamlit_app.py - Exercise 3: Streamlit Interface (Using Flask API)
Features:
1. Student name input
2. Study-hours and attendance inputs
3. Predict button that calls the Flask API
4. Result message with student name
5. Explanation of how prediction was generated
6. Comparison chart comparing study hours vs prediction
"""
import streamlit as st
import requests
import joblib
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="Student Pass Predictor", page_icon="🎓", layout="wide")

API_URL = "http://127.0.0.1:5000/predict"
LOCAL_MODEL_PATH = os.path.join(os.path.dirname(__file__), "student_pass_model.joblib")

# Reference comparison dataset
HISTORICAL_DATA = pd.DataFrame({
    'study_hours': [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0, 8.5, 9.0],
    'attendance_percentage': [50, 55, 60, 65, 70, 60, 75, 80, 70, 85, 90, 85, 90, 95, 98],
    'passed': ['Fail', 'Fail', 'Fail', 'Fail', 'Fail', 'Fail', 'Pass', 'Pass', 'Pass', 'Pass', 'Pass', 'Pass', 'Pass', 'Pass', 'Pass']
})

st.title("🎓 Student Pass/Fail Prediction Portal")
st.markdown("Deploying Machine Learning models via **Flask REST API** and **Streamlit**.")
st.divider()

col1, col2 = st.columns([1, 1], gap="large")

with col1:
    st.subheader("📝 Student Profile & Inputs")
    student_name = st.text_input("Student Name", value="Alex Morgan")
    study_hours = st.number_input("Daily Study Hours", min_value=0.0, max_value=24.0, value=5.0, step=0.5)
    attendance_percentage = st.slider("Attendance Percentage (%)", min_value=0, max_value=100, value=75)

    use_api = st.checkbox("Connect via Flask REST API (Exercise Standard)", value=True)
    predict_btn = st.button("🚀 Predict Student Outcome", type="primary", use_container_width=True)

    if predict_btn:
        if not student_name.strip():
            st.warning("Please enter a student name.")
        else:
            prediction = None
            prob = 0.5
            source = ""

            # Attempt via Flask API
            if use_api:
                try:
                    payload = {
                        "study_hours": float(study_hours),
                        "attendance_percentage": float(attendance_percentage)
                    }
                    res = requests.post(API_URL, json=payload, timeout=3)
                    if res.status_code == 200:
                        data = res.json()
                        prediction = data.get("prediction")
                        prob = data.get("pass_probability", 0.5)
                        source = "Flask API (HTTP 200 OK)"
                    else:
                        st.error(f"Flask Error: {res.json().get('error', res.text)}")
                except requests.exceptions.RequestException:
                    st.warning("Flask API not reachable at http://127.0.0.1:5000. Using local model fallback.")
                    if os.path.exists(LOCAL_MODEL_PATH):
                        m = joblib.load(LOCAL_MODEL_PATH)
                        inp = np.array([[study_hours, attendance_percentage]])
                        prediction = "Pass" if m.predict(inp)[0] == 1 else "Fail"
                        prob = m.predict_proba(inp)[0][1]
                        source = "Local Model Fallback (joblib.load)"
            else:
                if os.path.exists(LOCAL_MODEL_PATH):
                    m = joblib.load(LOCAL_MODEL_PATH)
                    inp = np.array([[study_hours, attendance_percentage]])
                    prediction = "Pass" if m.predict(inp)[0] == 1 else "Fail"
                    prob = m.predict_proba(inp)[0][1]
                    source = "Local Model (joblib.load)"

            if prediction:
                st.markdown("---")
                if prediction == "Pass":
                    st.success(f"### 🎉 Congratulations {student_name}! You are predicted to PASS.")
                else:
                    st.error(f"### ⚠️ Warning {student_name}: You are predicted to FAIL.")

                m1, m2, m3 = st.columns(3)
                m1.metric("Predicted Outcome", prediction)
                m2.metric("Pass Probability", f"{prob * 100:.1f}%")
                m3.metric("Engine Source", source)

                with st.expander("ℹ️ How was this prediction generated?", expanded=True):
                    st.write(f"""
                    **Decision Explanation:**
                    - Inputs: **{study_hours} hrs/day**, **{attendance_percentage}% attendance**.
                    - Model: Scikit-Learn Logistic Regression saved via **joblib**.
                    - Calculation: \( z = w_1 \cdot \text{{Hours}} + w_2 \cdot \text{{Attendance}} + b \).
                    - Probability = \( 1 / (1 + e^{{-z}}) \). When probability >= 50%, student passes.
                    """)

with col2:
    st.subheader("📊 Comparative Chart: Study Hours vs Prediction")
    st.caption("Visualizing the student relative to historical pass/fail distribution.")

    fig, ax = plt.subplots(figsize=(6, 4))
    passed_df = HISTORICAL_DATA[HISTORICAL_DATA['passed'] == 'Pass']
    failed_df = HISTORICAL_DATA[HISTORICAL_DATA['passed'] == 'Fail']

    ax.scatter(passed_df['study_hours'], passed_df['attendance_percentage'], color='#10b981', label='Historical Pass', s=50)
    ax.scatter(failed_df['study_hours'], failed_df['attendance_percentage'], color='#ef4444', label='Historical Fail', s=50)
    ax.scatter([study_hours], [attendance_percentage], color='#3b82f6', edgecolor='black', s=180, marker='*', label=f'Current: {student_name}', zorder=5)

    x_line = np.linspace(0, 10, 100)
    y_line = np.clip((65 - x_line * 7.5) / 0.55, 0, 100)
    ax.plot(x_line, y_line, '--', color='#6b7280', label='Decision Boundary')

    ax.set_xlim(0, 10)
    ax.set_ylim(40, 105)
    ax.set_xlabel("Daily Study Hours")
    ax.set_ylabel("Attendance Percentage (%)")
    ax.set_title("Student Decision Boundary Chart")
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.5)

    st.pyplot(fig)

## 📥 Step 5: Download Completed Unified Folder (.ZIP) from Colab

In [ ]:
from google.colab import files\n%cd ..\n!zip -r student_prediction_project.zip student_prediction_project\nfiles.download('student_prediction_project.zip')\nprint('✅ Downloaded student_prediction_project.zip!')